In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

KAGGLE_TRAINING_CSV = 'Training.csv'
YOUR_ORIGINAL_DISEASES_SYMPTOMS_CSV = 'Diseases_Symptoms.csv'

treatment_dict = {}

try:
    df_model_data = pd.read_csv(KAGGLE_TRAINING_CSV)

    if 'prognosis' in df_model_data.columns:
        df_model_data.rename(columns={'prognosis': 'name'}, inplace=True)
    else:
        raise ValueError(f"The dataset '{KAGGLE_TRAINING_CSV}' must contain a 'prognosis' column for disease names.")

    df_model_data.columns = [col.lower().strip() for col in df_model_data.columns]

    if 'name' not in df_model_data.columns:
        raise ValueError(f"The 'name' column was not found or incorrectly processed in '{KAGGLE_TRAINING_CSV}'.")

    df_original_treatments = pd.read_csv(YOUR_ORIGINAL_DISEASES_SYMPTOMS_CSV)

    if 'Code' in df_original_treatments.columns:
        df_original_treatments.rename(columns={'Code': 'Name'}, inplace=True)
    elif 'Name' not in df_original_treatments.columns:
        raise ValueError(f"The treatment dataset must contain either a 'Code' or 'Name' column.")

    if 'Treatments' not in df_original_treatments.columns:
        raise ValueError(f"The treatment dataset must contain a 'Treatments' column.")

    df_original_treatments['Name'] = df_original_treatments['Name'].astype(str).str.lower()
    df_original_treatments['Treatments'] = df_original_treatments['Treatments'].fillna('').astype(str).str.lower()

    treatment_dict = dict(zip(df_original_treatments['Name'], df_original_treatments['Treatments']))

except FileNotFoundError as e:
    print(f"Error: Required CSV file not found: {e}.")
    exit()
except ValueError as e:
    print(f"Error processing dataset: {e}")
    exit()
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    exit()

X = df_model_data.drop(columns=['name'])
y = df_model_data['name']

all_symptoms = list(X.columns)
X.fillna(0, inplace=True)

disease_counts = y.value_counts()
single_entry_diseases = disease_counts[disease_counts < 2].index.tolist()

if single_entry_diseases:
    indices_to_keep = y[~y.isin(single_entry_diseases)].index
    X = X.loc[indices_to_keep]
    y = y.loc[indices_to_keep]

    if X.empty or y.empty:
        print("Error: No data remains after filtering.")
        exit()

if X.empty or y.empty:
    print("Error: No data or labels found.")
    exit()

if X.shape[1] == 0:
    print("Error: No symptom columns found.")
    exit()

model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')

n_classes = len(y.unique())
total_samples = len(X)
min_samples_for_stratification = n_classes * 2

if total_samples < min_samples_for_stratification:
    test_size_param = max(2, int(total_samples * 0.2))
    if total_samples - test_size_param < n_classes:
        test_size_param = max(2, total_samples - n_classes)
        if test_size_param < 1: test_size_param = 1

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size_param, random_state=42)
else:
    target_test_size_fraction = 0.2
    calculated_test_samples = max(n_classes, int(total_samples * target_test_size_fraction))

    if total_samples - calculated_test_samples < n_classes:
        calculated_test_samples = total_samples - n_classes

    test_size_param = calculated_test_samples

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size_param, random_state=42, stratify=y)

if X_train.empty or y_train.empty:
    print("Error: Training data is empty after split.")
    exit()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

def diagnose_patient(name, age, gender, symptoms_input):
    symptoms_input = [s.strip().lower() for s in symptoms_input]

    input_vector_dict = {symptom_col: 0 for symptom_col in all_symptoms}
    for symptom in symptoms_input:
        if symptom in input_vector_dict:
            input_vector_dict[symptom] = 1
        else:
            print(f"Warning: Symptom '{symptom}' not recognized.")

    input_vector = [input_vector_dict[symptom_col] for symptom_col in all_symptoms]

    if not any(input_vector):
        predicted_disease = "No specific disease identified."
        treatment_plan = "Consult a medical professional."
    else:
        try:
            probabilities = model.predict_proba([input_vector])[0]
            disease_probabilities = list(zip(model.classes_, probabilities))
            disease_probabilities.sort(key=lambda x: x[1], reverse=True)

            predicted_disease = disease_probabilities[0][0]
            predicted_probability = disease_probabilities[0][1]

            common_mild_symptoms = {
                'fever', 'headache', 'cough', 'sore_throat', 'runny_nose', 'fatigue',
                'body_aches', 'sneezing', 'dizziness', 'chills', 'mild_fever', 'sweating',
                'vomiting', 'diarrhoea', 'abdominal_pain', 'irritability'
            }

            severe_diseases = {
                "aids", "paralysis (brain hemorrhage)", "jaundice", "malaria", "dengue",
                "typhoid", "hepatitis b", "hepatitis c", "hepatitis d", "pneumonia",
                "peptic ulcer diseae", "hypertension", "diabetes ", "bronchial asthma",
                "cervical spondylosis", "tuberculosis", "heart attack", "stroke",
                "chicken pox", "arthritis", "migraine", "tuberculosis", "encephalitis",
                "lymphoma", "leukemia", "adrenal cancer", "breast cancer", "testicular cancer",
                "endometrial cancer", "esophageal cancer", "liver cancer", "bone cancer"
            }

            input_symptoms_set = set(symptoms_input)
            is_predominantly_mild = len(input_symptoms_set.intersection(common_mild_symptoms)) >= (len(input_symptoms_set) * 0.8)

            if (is_predominantly_mild and predicted_disease.lower() in severe_diseases and predicted_probability < 0.5):
                predicted_disease = "Symptoms suggest a common mild illness."
                treatment_plan = "Rest, hydration, OTC medications. If symptoms worsen, seek professional care."
            else:
                treatment_plan = treatment_dict.get(predicted_disease.lower(), "Consult a medical professional.")

            print("\nTop 3 possible diagnoses:")
            for disease, prob in disease_probabilities[:3]:
                print(f"   - {disease} (Confidence: {prob:.2f})")
        except Exception as e:
            predicted_disease = "Error during prediction."
            treatment_plan = f"Unable to predict due to issue: {e}."

    print("\n" + "=="*100)
    print("       AI-Based Medical Diagnosis Report")
    print("=="*100)
    print(f"Name       : {name}")
    print(f"Age        : {age}")
    print(f"Gender     : {gender}")
    print(f"Symptoms   : {', '.join(symptoms_input)}")
    print("--" * 100)
    print(f"Predicted Disease : {predicted_disease}")
    print(f"Suggested Treatment: {treatment_plan}")
    print("=="*100 + "\n")

if __name__ == "__main__":
    print("--"*5 + "Welcome to the AI-Based Medical Diagnosis Assistant!" + "--"*5)
    name = input("Enter your name: ")
    age = input("Enter your age: ")
    gender = input("Enter your gender (e.g., Male, Female, Other): ")

    print("\nEnter symptoms separated by commas (e.g., itching,skin_rash,continuous_sneezing):")
    symptoms_raw = input("Symptoms: ")
    symptoms_list = [s.strip().lower() for s in symptoms_raw.split(',')]

    diagnose_patient(name, age, gender, symptoms_list)